In [ ]:
# =============================================================================
# ERASMUS MOBILITY DATA: LLM VALIDATION & MANUAL CORRECTIONS
# =============================================================================
# 
# This notebook validates and corrects geocoding errors through:
# 1. LLM-based coordinate validation (DeepSeek-R1:1.5b via Ollama)
# 2. Manual corrections dictionary (525 known errors)
# 3. Capital city fallback for missing/invalid locations
#
# INPUT:  erasmus_with_nuts3.csv (output from notebook 01)
# OUTPUT: erasmus_with_nuts3_corrected.csv - validated and corrected data
#
# REQUIREMENTS:
# - Ollama installed and running (https://ollama.ai)
# - DeepSeek-R1:1.5b model pulled (ollama pull deepseek-r1:1.5b)
# - NUTS3 shapefile from notebook 01
#
# ESTIMATED RUNTIME: 30-45 minutes (depending on number of problematic locations)
# =============================================================================

In [ ]:
# =============================================================================
# CELL 1: Import Libraries
# =============================================================================

import pandas as pd
import geopandas as gpd
import json
import time
import subprocess
from tqdm import tqdm
from openai import OpenAI
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries loaded successfully!")

In [ ]:
# =============================================================================
# CELL 2: Configuration - USER INPUT REQUIRED
# =============================================================================

# ==========================
# CONFIGURE THESE PATHS:
# ==========================

# Path to geocoded data (output from notebook 01)
ERASMUS_GEOCODED_PATH = '/path/to/erasmus_with_nuts3.csv'

# Path to NUTS3 shapefile (same as notebook 01)
NUTS3_SHAPEFILE_PATH = '/path/to/NUTS_RG_01M_2021_3035.shp'

# Output directory
OUTPUT_DIR = '/path/to/output/directory/'

# ==========================
# LLM CONFIGURATION:
# ==========================

# Ollama model to use for validation
LLM_MODEL = "deepseek-r1:1.5b"

# Ollama API endpoint
OLLAMA_BASE_URL = 'http://localhost:11434/v1'

# Batch size for LLM processing (saves progress incrementally)
LLM_BATCH_SIZE = 5

print("=" * 80)
print("CONFIGURATION")
print("=" * 80)
print(f"Input data:  {ERASMUS_GEOCODED_PATH}")
print(f"NUTS3 shapefile: {NUTS3_SHAPEFILE_PATH}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"LLM model:   {LLM_MODEL}")
print(f"Batch size:  {LLM_BATCH_SIZE}")
print("=" * 80)

In [ ]:
# =============================================================================
# CELL 3: Load Data and Assess Current Coverage
# =============================================================================

print("=" * 80)
print("LOADING DATA AND ASSESSING NUTS3 COVERAGE")
print("=" * 80)

# Load geocoded dataset
df_erasmus = pd.read_csv(ERASMUS_GEOCODED_PATH)
print(f"✓ Dataset loaded: {len(df_erasmus):,} records")

# Load NUTS3 shapefile
nuts3 = gpd.read_file(NUTS3_SHAPEFILE_PATH)
nuts3 = nuts3[nuts3['LEVL_CODE'] == 3].copy()
print(f"✓ NUTS3 shapefile loaded: {len(nuts3):,} regions")

# Current NUTS3 coverage
total_records = len(df_erasmus)
sending_nuts3 = df_erasmus['Sending_NUTS3'].notna().sum()
receiving_nuts3 = df_erasmus['Receiving_NUTS3'].notna().sum()
both_nuts3 = df_erasmus[
    (df_erasmus['Sending_NUTS3'].notna()) & 
    (df_erasmus['Receiving_NUTS3'].notna())
].shape[0]

print(f"\n📊 CURRENT NUTS3 COVERAGE:")
print(f"  Sending:   {sending_nuts3:,} / {total_records:,} ({sending_nuts3/total_records*100:.1f}%)")
print(f"  Receiving: {receiving_nuts3:,} / {total_records:,} ({receiving_nuts3/total_records*100:.1f}%)")
print(f"  Both:      {both_nuts3:,} / {total_records:,} ({both_nuts3/total_records*100:.1f}%)")

In [ ]:
# =============================================================================
# CELL 4: Identify Problematic Locations for LLM Validation
# =============================================================================

print("\n" + "=" * 80)
print("IDENTIFYING PROBLEMATIC LOCATIONS")
print("=" * 80)

# Get locations that were geocoded but don't have NUTS3
# (likely wrong country or bad coordinates)
missing_sending = df_erasmus[
    (df_erasmus['Sending_Lat'].notna()) & 
    (df_erasmus['Sending_NUTS3'].isna())
][['Sending City', 'Sending Country', 'Sending_Lat', 'Sending_Lon']].drop_duplicates()

missing_receiving = df_erasmus[
    (df_erasmus['Receiving_Lat'].notna()) & 
    (df_erasmus['Receiving_NUTS3'].isna())
][['Receiving City', 'Receiving Country', 'Receiving_Lat', 'Receiving_Lon']].drop_duplicates()

# Rename for consistency
missing_sending.columns = ['city', 'country', 'latitude', 'longitude']
missing_receiving.columns = ['city', 'country', 'latitude', 'longitude']

# Combine and deduplicate
all_missing = pd.concat([missing_sending, missing_receiving]).drop_duplicates()

# Filter out invalid entries
all_missing = all_missing[
    (all_missing['city'].str.len() > 1) &  # Remove single letters
    (all_missing['city'] != '-')  # Remove dashes
]

print(f"✓ Identified {len(all_missing):,} unique problematic locations")
print(f"\nTop 10 countries with problematic locations:")
for country, count in all_missing['country'].value_counts().head(10).items():
    print(f"  {country:40s} {count:4,} locations")

In [ ]:
# =============================================================================
# CELL 5: Setup Ollama LLM
# =============================================================================

print("\n" + "=" * 80)
print("SETTING UP OLLAMA LLM")
print("=" * 80)

# Check if Ollama is installed
try:
    result = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
    print(f"✓ Ollama installed: {result.stdout.strip()}")
except FileNotFoundError:
    print("❌ Ollama not found. Please install from https://ollama.ai")
    print("   Then run: ollama serve")
    raise SystemExit(1)

# Pull the model if not already available
print(f"\nPulling model: {LLM_MODEL}...")
try:
    result = subprocess.run(['ollama', 'pull', LLM_MODEL], 
                          capture_output=True, text=True, check=True)
    print(f"✓ Model ready: {LLM_MODEL}")
except subprocess.CalledProcessError as e:
    print(f"❌ Error pulling model: {e.stderr}")
    raise

# Initialize Ollama client
client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key='ollama',  # Required but unused
)

# Test connection
try:
    models = client.models.list()
    available_models = [m.id for m in models.data]
    
    if LLM_MODEL in available_models:
        print(f"✓ Connected to Ollama - {LLM_MODEL} available")
    else:
        print(f"⚠️  {LLM_MODEL} not found in: {available_models}")
        raise SystemExit(1)
        
except Exception as e:
    print(f"❌ Error connecting to Ollama: {e}")
    print("\nMake sure Ollama is running: ollama serve")
    raise SystemExit(1)

In [ ]:
# =============================================================================
# CELL 6: Define LLM Validation Function
# =============================================================================

SYSTEM_MESSAGE = """You are a coordinate validator. Return ONLY valid JSON, no explanation.

Given a city, country, and coordinates, return corrected coordinates if wrong, or same if correct.

If you cannot identify the city, use the country's capital coordinates.

Format:
{"corrected_latitude": number, "corrected_longitude": number, "correction_method": "validated_correct|corrected_location|fallback_to_capital"}

Example:
Input: Heraklion, Greece, (31.30, 30.12)
Output: {"corrected_latitude": 35.3387, "corrected_longitude": 25.1442, "correction_method": "corrected_location"}"""

def get_corrected_coordinates(city, country, current_lat, current_lon):
    """
    LLM-based coordinate validation and correction.
    
    Parameters:
    -----------
    city : str
        City name
    country : str
        Country name
    current_lat : float
        Current latitude
    current_lon : float
        Current longitude
    
    Returns:
    --------
    dict : {'corrected_latitude', 'corrected_longitude', 'correction_method'}
    """
    user_message = f"{city}, {country}: ({current_lat}, {current_lon})"

    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[
                {"role": "system", "content": SYSTEM_MESSAGE},
                {"role": "user", "content": user_message}
            ],
            temperature=0.0,
            max_tokens=150
        )
        
        content = response.choices[0].message.content.strip()
        
        # Extract JSON (handle markdown code blocks)
        if "```json" in content:
            content = content.split("```json")[1].split("```")[0].strip()
        elif "```" in content:
            content = content.split("```")[1].split("```")[0].strip()
        
        # Find JSON object
        start = content.find('{')
        end = content.rfind('}') + 1
        if start != -1 and end > start:
            json_str = content[start:end]
            result = json.loads(json_str)
            
            if "corrected_latitude" in result and "corrected_longitude" in result:
                return result
            else:
                raise ValueError("Missing required keys in LLM response")
        else:
            raise ValueError("No JSON found in LLM response")
            
    except Exception as e:
        # On error, return original coordinates
        return {
            "corrected_latitude": current_lat,
            "corrected_longitude": current_lon,
            "correction_method": "error_fallback"
        }

print("✓ LLM validation function defined")

In [ ]:
# =============================================================================
# CELL 7: Run LLM Validation (with Progress Saving)
# =============================================================================

print("\n" + "=" * 80)
print(f"RUNNING LLM VALIDATION ON {len(all_missing):,} LOCATIONS")
print("=" * 80)
print(f"Processing in batches of {LLM_BATCH_SIZE}")
print("Progress saved after each batch\n")

# Prepare location list
missing_locations = []
for idx, row in all_missing.iterrows():
    location = {
        "city": row['city'],
        "country": row['country'],
        "current_latitude": float(row['latitude']),
        "current_longitude": float(row['longitude']),
        "corrected_latitude": None,
        "corrected_longitude": None,
        "correction_method": None
    }
    missing_locations.append(location)

# Load existing progress if available
progress_file = f"{OUTPUT_DIR}/llm_corrections_progress.json"
try:
    with open(progress_file, 'r', encoding='utf-8') as f:
        corrected = json.load(f)
    print(f"✓ Loaded {len(corrected)} already corrected locations")
    
    # Filter out already processed
    processed_keys = {(c['city'], c['country']) for c in corrected}
    remaining = [loc for loc in missing_locations 
                 if (loc['city'], loc['country']) not in processed_keys]
except FileNotFoundError:
    corrected = []
    remaining = missing_locations.copy()

print(f"Remaining to process: {len(remaining)}\n")

# Process in batches
total_batches = (len(remaining) + LLM_BATCH_SIZE - 1) // LLM_BATCH_SIZE

for batch_num in range(0, len(remaining), LLM_BATCH_SIZE):
    batch = remaining[batch_num:batch_num + LLM_BATCH_SIZE]
    
    print(f"Batch {batch_num//LLM_BATCH_SIZE + 1}/{total_batches} " +
          f"(locations {batch_num+1}-{min(batch_num+LLM_BATCH_SIZE, len(remaining))})")
    
    for location in batch:
        print(f"  {location['city']:30s} {location['country']:30s}", end=" ... ")
        
        result = get_corrected_coordinates(
            location['city'],
            location['country'],
            location['current_latitude'],
            location['current_longitude']
        )
        
        location['corrected_latitude'] = result['corrected_latitude']
        location['corrected_longitude'] = result['corrected_longitude']
        location['correction_method'] = result.get('correction_method', 'unknown')
        
        corrected.append(location)
        print("✓")
    
    # Save progress after each batch
    with open(progress_file, 'w', encoding='utf-8') as f:
        json.dump(corrected, f, indent=2, ensure_ascii=False)
    
    print(f"  💾 Progress saved: {len(corrected)}/{len(missing_locations)}\n")

print(f"✅ LLM VALIDATION COMPLETE: {len(corrected):,} locations")

# Summary of correction methods
print("\n📊 Correction methods:")
methods = {}
for loc in corrected:
    m = loc.get('correction_method', 'unknown')
    methods[m] = methods.get(m, 0) + 1

for method, count in sorted(methods.items(), key=lambda x: x[1], reverse=True):
    print(f"  {method:25s} {count:6,}")

In [ ]:
# =============================================================================
# CELL 8: Apply Manual Corrections Dictionary (525 Known Errors)
# =============================================================================

print("\n" + "=" * 80)
print("APPLYING MANUAL CORRECTIONS DICTIONARY")
print("=" * 80)

# Dictionary of 525 manually verified corrections
MANUAL_CORRECTIONS = {
    # Spain - major fixes
    ('Vitoria', 'ES - Spain'): (42.8467, -2.6716),
    ('VITORIA', 'ES - Spain'): (42.8467, -2.6716),
    ('Vittoria', 'ES - Spain'): (42.8467, -2.6716),
    ('Las Palmas', 'ES - Spain'): (28.1235, -15.4363),
    ('LAS PALMAS', 'ES - Spain'): (28.1235, -15.4363),
    ('Las Palmas GC', 'ES - Spain'): (28.1235, -15.4363),
    ('Málag', 'ES - Spain'): (36.7213, -4.4214),
    ('Nálaga', 'ES - Spain'): (36.7213, -4.4214),
    ('EL PUERTO DE SANTA MARÍA34', 'ES - Spain'): (36.5995, -6.2281),
    ('Comarruga', 'ES - Spain'): (41.1833, 1.5167),
    ('Sala al Jadida', 'ES - Spain'): (35.8939, -5.3156),
    ('TBILISI', 'ES - Spain'): (40.4168, -3.7038),
    
    # Greece - major fixes
    ('Kos', 'EL - Greece'): (36.8933, 27.2886),
    ('KOS', 'EL - Greece'): (36.8933, 27.2886),
    ('Corfu', 'EL - Greece'): (39.6243, 19.9217),
    ('CORFU', 'EL - Greece'): (39.6243, 19.9217),
    ('Heraklion', 'EL - Greece'): (35.3387, 25.1442),
    ('HERAKLION', 'EL - Greece'): (35.3387, 25.1442),
    ('KALAMATA', 'EL - Greece'): (37.0392, 22.1142),
    ('Kalamata', 'EL - Greece'): (37.0392, 22.1142),
    ('Καλαμάτα', 'EL - Greece'): (37.0392, 22.1142),
    ('MELIKI', 'EL - Greece'): (40.5167, 22.2000),
    ('Meliki', 'EL - Greece'): (40.5167, 22.2000),
    ('Haidari', 'EL - Greece'): (38.0167, 23.6667),
    ('HAIDARI', 'EL - Greece'): (38.0167, 23.6667),
    ('Velo,  Korinthia', 'EL - Greece'): (38.0000, 22.7500),
    ('TRIPOLIS', 'EL - Greece'): (37.5089, 22.3778),
    ('Tripolis', 'EL - Greece'): (37.5089, 22.3778),
    ('TRIPOLI', 'EL - Greece'): (37.5089, 22.3778),
    ('Τρίπολη', 'EL - Greece'): (37.5089, 22.3778),
    ('sabadell', 'EL - Greece'): (37.9838, 23.7275),
    ('PATRAS', 'EL - Greece'): (38.2466, 21.7346),
    ('patras', 'EL - Greece'): (38.2466, 21.7346),
    ('Kavala', 'EL - Greece'): (40.9364, 24.4019),
    ('Rhodes', 'EL - Greece'): (36.4341, 28.2176),
    ('RHODES', 'EL - Greece'): (36.4341, 28.2176),
    ('rhodes', 'EL - Greece'): (36.4341, 28.2176),
    ('FLORINA', 'EL - Greece'): (40.7828, 21.4094),
    ('SPARTA', 'EL - Greece'): (37.0742, 22.4303),
    
    # NOTE: Full dictionary contains 525 entries
    # Add remaining entries here from your CORRECT_COORDS dictionary
    # ... (truncated for brevity in example)
}

print(f"Manual corrections dictionary: {len(MANUAL_CORRECTIONS)} entries")

# Apply manual corrections
corrections_applied = 0

for (city, country), (lat, lon) in MANUAL_CORRECTIONS.items():
    # Apply to Sending
    mask = (df_erasmus['Sending City'] == city) & (df_erasmus['Sending Country'] == country)
    if mask.any():
        df_erasmus.loc[mask, 'Sending_Lat'] = lat
        df_erasmus.loc[mask, 'Sending_Lon'] = lon
        corrections_applied += mask.sum()
    
    # Apply to Receiving
    mask = (df_erasmus['Receiving City'] == city) & (df_erasmus['Receiving Country'] == country)
    if mask.any():
        df_erasmus.loc[mask, 'Receiving_Lat'] = lat
        df_erasmus.loc[mask, 'Receiving_Lon'] = lon
        corrections_applied += mask.sum()

print(f"✓ Applied {corrections_applied:,} manual corrections")

In [ ]:
# =============================================================================
# CELL 9: Apply LLM Corrections to Main Dataset
# =============================================================================

print("\n" + "=" * 80)
print("APPLYING LLM CORRECTIONS TO MAIN DATASET")
print("=" * 80)

# Load LLM corrections
with open(progress_file, 'r', encoding='utf-8') as f:
    llm_corrections = json.load(f)

# Create lookup dictionary
correction_lookup = {}
for loc in llm_corrections:
    key = (loc['city'], loc['country'])
    correction_lookup[key] = {
        'lat': loc['corrected_latitude'],
        'lon': loc['corrected_longitude'],
        'method': loc['correction_method']
    }

print(f"LLM corrections: {len(correction_lookup)} entries")

# Apply LLM corrections
llm_corrections_applied = 0

for idx, row in df_erasmus.iterrows():
    # Sending
    key = (row['Sending City'], row['Sending Country'])
    if key in correction_lookup:
        correction = correction_lookup[key]
        df_erasmus.at[idx, 'Sending_Lat'] = correction['lat']
        df_erasmus.at[idx, 'Sending_Lon'] = correction['lon']
        llm_corrections_applied += 1
    
    # Receiving
    key = (row['Receiving City'], row['Receiving Country'])
    if key in correction_lookup:
        correction = correction_lookup[key]
        df_erasmus.at[idx, 'Receiving_Lat'] = correction['lat']
        df_erasmus.at[idx, 'Receiving_Lon'] = correction['lon']
        llm_corrections_applied += 1

print(f"✓ Applied {llm_corrections_applied:,} LLM corrections")

In [ ]:
# =============================================================================
# CELL 10: Capital City Fallback for Remaining Missing Locations
# =============================================================================

print("\n" + "=" * 80)
print("APPLYING CAPITAL CITY FALLBACK")
print("=" * 80)

# Capital cities with pre-validated NUTS3 codes
CAPITALS_WITH_NUTS = {
    'AT - Austria': (48.2082, 16.3738, 'AT130'),
    'BE - Belgium': (50.8503, 4.3517, 'BE100'),
    'BG - Bulgaria': (42.6977, 23.3219, 'BG411'),
    'CY - Cyprus': (35.1264, 33.4299, 'CY000'),
    'CZ - Czechia': (50.0755, 14.4378, 'CZ010'),
    'DE - Germany': (52.5200, 13.4050, 'DE300'),
    'DK - Denmark': (55.6761, 12.5683, 'DK012'),
    'EE - Estonia': (59.4370, 24.7536, 'EE001'),
    'EL - Greece': (37.9838, 23.7275, 'EL303'),
    'ES - Spain': (40.4168, -3.7038, 'ES300'),
    'FI - Finland': (60.1695, 24.9354, 'FI1B1'),
    'FR - France': (48.8566, 2.3522, 'FR101'),
    'HR - Croatia': (45.8150, 15.9819, 'HR050'),
    'HU - Hungary': (47.4979, 19.0402, 'HU110'),
    'IE - Ireland': (53.3498, -6.2603, 'IE061'),
    'IS - Iceland': (64.1466, -21.9426, 'IS001'),
    'IT - Italy': (41.9028, 12.4964, 'ITI43'),
    'LT - Lithuania': (54.6872, 25.2797, 'LT011'),
    'LU - Luxembourg': (49.6116, 6.1319, 'LU000'),
    'LV - Latvia': (56.9496, 24.1052, 'LV006'),
    'MT - Malta': (35.8989, 14.5146, 'MT001'),
    'NL - Netherlands': (52.3676, 4.9041, 'NL329'),
    'NO - Norway': (59.9139, 10.7522, 'NO011'),
    'PL - Poland': (52.2297, 21.0122, 'PL911'),
    'PT - Portugal': (38.7223, -9.1393, 'PT170'),
    'RO - Romania': (44.4268, 26.1025, 'RO321'),
    'SE - Sweden': (59.3293, 18.0686, 'SE110'),
    'SI - Slovenia': (46.0569, 14.5058, 'SI041'),
    'SK - Slovakia': (48.1486, 17.1077, 'SK010'),
    'TR - Turkey': (39.9334, 32.8597, 'TR510'),
    'UK - United Kingdom': (51.5074, -0.1278, 'UKI11'),
}

print(f"Capital city fallbacks: {len(CAPITALS_WITH_NUTS)} countries")

fallbacks_applied = 0

for country, (lat, lon, nuts3) in CAPITALS_WITH_NUTS.items():
    # Sending: missing coordinates or NUTS3
    mask = (
        (df_erasmus['Sending Country'] == country) & 
        (df_erasmus['Sending_NUTS3'].isna() | df_erasmus['Sending_Lat'].isna())
    )
    count = mask.sum()
    if count > 0:
        df_erasmus.loc[mask, 'Sending_Lat'] = lat
        df_erasmus.loc[mask, 'Sending_Lon'] = lon
        df_erasmus.loc[mask, 'Sending_NUTS3'] = nuts3
        fallbacks_applied += count
    
    # Receiving: missing coordinates or NUTS3
    mask = (
        (df_erasmus['Receiving Country'] == country) & 
        (df_erasmus['Receiving_NUTS3'].isna() | df_erasmus['Receiving_Lat'].isna())
    )
    count = mask.sum()
    if count > 0:
        df_erasmus.loc[mask, 'Receiving_Lat'] = lat
        df_erasmus.loc[mask, 'Receiving_Lon'] = lon
        df_erasmus.loc[mask, 'Receiving_NUTS3'] = nuts3
        fallbacks_applied += count

print(f"✓ Applied {fallbacks_applied:,} capital city fallbacks")

In [ ]:
# =============================================================================
# CELL 11: Re-match All Locations to NUTS3 with Corrected Coordinates
# =============================================================================

print("\n" + "=" * 80)
print("RE-MATCHING WITH CORRECTED COORDINATES")
print("=" * 80)

# Re-match Sending locations
print("Re-matching Sending locations...")
sending_valid = df_erasmus[df_erasmus['Sending_Lat'].notna()].copy()
sending_gdf = gpd.GeoDataFrame(
    sending_valid,
    geometry=gpd.points_from_xy(sending_valid['Sending_Lon'], sending_valid['Sending_Lat']),
    crs="EPSG:4326"
)
sending_gdf = sending_gdf.to_crs(nuts3.crs)
sending_matched = gpd.sjoin(
    sending_gdf,
    nuts3[['NUTS_ID', 'NUTS_NAME', 'geometry']],
    how='left',
    predicate='within'
)
df_erasmus['Sending_NUTS3'] = df_erasmus.index.map(
    dict(zip(sending_matched.index, sending_matched['NUTS_ID']))
)
df_erasmus['Sending_NUTS3_Name'] = df_erasmus.index.map(
    dict(zip(sending_matched.index, sending_matched['NUTS_NAME']))
)
print(f"✓ Sending NUTS3: {df_erasmus['Sending_NUTS3'].notna().sum():,}")

# Re-match Receiving locations
print("Re-matching Receiving locations...")
receiving_valid = df_erasmus[df_erasmus['Receiving_Lat'].notna()].copy()
receiving_gdf = gpd.GeoDataFrame(
    receiving_valid,
    geometry=gpd.points_from_xy(receiving_valid['Receiving_Lon'], receiving_valid['Receiving_Lat']),
    crs="EPSG:4326"
)
receiving_gdf = receiving_gdf.to_crs(nuts3.crs)
receiving_matched = gpd.sjoin(
    receiving_gdf,
    nuts3[['NUTS_ID', 'NUTS_NAME', 'geometry']],
    how='left',
    predicate='within'
)
df_erasmus['Receiving_NUTS3'] = df_erasmus.index.map(
    dict(zip(receiving_matched.index, receiving_matched['NUTS_ID']))
)
df_erasmus['Receiving_NUTS3_Name'] = df_erasmus.index.map(
    dict(zip(receiving_matched.index, receiving_matched['NUTS_NAME']))
)
print(f"✓ Receiving NUTS3: {df_erasmus['Receiving_NUTS3'].notna().sum():,}")

In [ ]:
# =============================================================================
# CELL 12: Save Corrected Dataset and Generate Final Summary
# =============================================================================

print("\n" + "=" * 80)
print("SAVING CORRECTED DATASET")
print("=" * 80)

# Save corrected dataset
output_path = f"{OUTPUT_DIR}/erasmus_with_nuts3_corrected.csv"
df_erasmus.to_csv(output_path, index=False)

print(f"✓ Corrected dataset saved: erasmus_with_nuts3_corrected.csv")
print(f"  Location: {output_path}")

# Final coverage statistics
total_records = len(df_erasmus)
sending_nuts3_final = df_erasmus['Sending_NUTS3'].notna().sum()
receiving_nuts3_final = df_erasmus['Receiving_NUTS3'].notna().sum()
both_nuts3_final = df_erasmus[
    (df_erasmus['Sending_NUTS3'].notna()) & 
    (df_erasmus['Receiving_NUTS3'].notna())
].shape[0]

print("\n" + "=" * 80)
print("FINAL NUTS3 COVERAGE")
print("=" * 80)
print(f"\nTotal records: {total_records:,}")
print(f"\n📊 FINAL COVERAGE:")
print(f"  Sending NUTS3:   {sending_nuts3_final:,} / {total_records:,} " +
      f"({sending_nuts3_final/total_records*100:.2f}%)")
print(f"  Receiving NUTS3: {receiving_nuts3_final:,} / {total_records:,} " +
      f"({receiving_nuts3_final/total_records*100:.2f}%)")
print(f"  Both present:    {both_nuts3_final:,} / {total_records:,} " +
      f"({both_nuts3_final/total_records*100:.2f}%)")

# Improvement summary
print(f"\n📈 IMPROVEMENT:")
print(f"  Before:  {both_nuts3:,} ({both_nuts3/total_records*100:.1f}%)")
print(f"  After:   {both_nuts3_final:,} ({both_nuts3_final/total_records*100:.2f}%)")
print(f"  Gain:    +{both_nuts3_final - both_nuts3:,} records")

# Correction summary
print(f"\n🔧 CORRECTIONS APPLIED:")
print(f"  Manual corrections:  {corrections_applied:,}")
print(f"  LLM corrections:     {llm_corrections_applied:,}")
print(f"  Capital fallbacks:   {fallbacks_applied:,}")
print(f"  Total:               {corrections_applied + llm_corrections_applied + fallbacks_applied:,}")

print("\n" + "=" * 80)
print("✅ LLM VALIDATION AND CORRECTIONS COMPLETE")
print("=" * 80)